# RAG-Tutorial: Frag mich zur Geschichte von RAG!

Dieses Notebook baut die klassische RAG-Pipeline in fünf Schritten auf —
ohne Framework, nur mit der OpenAI-API, FAISS und NumPy direkt.
Einzige Ausnahme: das Chunking übernimmt der bewährte Splitter aus dem
kleinen Standalone-Paket `langchain-text-splitters`.

1. **LADEN** — Wissensdokument (`rag_geschichte.txt`) einlesen
2. **CHUNKING** — Dokument in überlappende Textblöcke zerlegen
3. **INDEXIEREN** — Blöcke einbetten und im FAISS-Vektorspeicher ablegen
4. **RETRIEVAL** — Zur Frage die ähnlichsten Blöcke suchen
5. **GENERATION** — Chat-Modell beantwortet die Frage NUR mit diesen Blöcken

Schritte 1–3 laufen **einmal** (Indexierungsphase),
Schritte 4–5 laufen **bei jeder Frage** neu (Abfragephase).

## Setup: Pakete importieren

Falls ein Import fehlschlägt: oben rechts den richtigen Kernel wählen oder
die Pakete nachinstallieren mit `pip install -r requirements.txt`.

In [1]:
import os

import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

/home/ml/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ml/miniconda3/envs/vllm/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Schritt 0: Konfiguration

Alle Einstellungen kommen aus der `.env`-Datei. Ohne `BASE_URL` wird direkt
die OpenAI-API benutzt; mit `BASE_URL` jeder OpenAI-kompatible Server
(z. B. ein lokaler Ollama-Server unter `http://localhost:11434/v1`).

Die drei Stellschrauben der Pipeline:

- `TOP_K` — wie viele Textblöcke pro Frage als Kontext verwendet werden
- `BLOCK_GROESSE` — Zielgröße eines Blocks in Zeichen
- `UEBERLAPPUNG` — so viele Zeichen teilen sich zwei Nachbarblöcke

In [ ]:
load_dotenv()

BASE_URL = os.getenv("BASE_URL")  # None => echtes OpenAI
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

WISSENSDATEI = "rag_geschichte.txt"
TOP_K = 3            # Wie viele Textblöcke pro Frage als Kontext verwendet werden.
BLOCK_GROESSE = 300  # Zielgröße eines Blocks in Zeichen
UEBERLAPPUNG = 60    # so viele Zeichen teilen sich zwei Nachbarblöcke (Obergrenze)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY fehlt (in der .env-Datei setzen)."

# Ein Client für alles: Embeddings UND Chat laufen über denselben Server.
# Den API-Schlüssel liest der Client selbst aus OPENAI_API_KEY.
client = OpenAI(base_url=BASE_URL)
print(f"Chat-Modell: {CHAT_MODEL} | Embedding-Modell: {EMBEDDING_MODEL}")

Chat-Modell: qwen2.5:14b | Embedding-Modell: bge-m3


## Schritt 1: Wissensdokument laden

Hier eine einfache Textdatei. In echten Projekten kommen an dieser Stelle
Parser für PDF, HTML usw. zum Einsatz.

In [7]:
with open(WISSENSDATEI, encoding="utf-8") as f:
    text = f.read()

print(f"{len(text)} Zeichen geladen. So beginnt das Dokument:\n")
print(text[:300], "...")

7869 Zeichen geladen. So beginnt das Dokument:

Die Geschichte der Retrieval-Augmented Generation (RAG)

Vorgeschichte: Suche vor den Sprachmodellen
Lange bevor es große Sprachmodelle gab, beschäftigte sich das Feld Information Retrieval (IR) damit, relevante Dokumente zu einer Suchanfrage zu finden. Klassische Verfahren wie TF-IDF (Term Frequenc ...


## Schritt 2: Chunking — das Dokument in Blöcke zerlegen

Warum? Ein Embedding pro **ganzem** Dokument wäre zu unscharf, und das
Chat-Modell soll später nur die relevanten Ausschnitte sehen.

Der `RecursiveCharacterTextSplitter` versucht, an sinnvollen Grenzen zu
trennen (erst Absätze, dann Zeilen, dann Wörter). Die Überlappung sorgt
dafür, dass kein Zusammenhang genau an einer Schnittkante verloren geht.

**`chunk_overlap` ist eine Obergrenze, keine Garantie:**

- An **Absatzgrenzen** gibt es *keine* Überlappung. Der Splitter zerlegt
  zuerst in Absätze und überlappt nur *innerhalb* eines Absatzes, der
  weiter zerlegt werden musste. Ist `BLOCK_GROESSE` größer als ein Absatz,
  ist jeder Block ein ganzer Absatz — und nichts überlappt sich.
  (Unsere Absätze sind 55–887 Zeichen lang, deshalb `BLOCK_GROESSE = 300`.)
- Die Überlappung rastet auf **Wortgrenzen** ein: geteilt werden also
  47 oder 59 Zeichen, nie exakt 60.
- Bei sehr kleiner `BLOCK_GROESSE` (z. B. 50) frisst schon ein einziges
  langes deutsches Wort die ganze Überlappung auf.


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=BLOCK_GROESSE,
    chunk_overlap=UEBERLAPPUNG,
)
bloecke = splitter.split_text(text)
print(f"Dokument in {len(bloecke)} Blöcke zerlegt.\n")
print("Block 0:\n")
print(bloecke[0])

Dokument in 198 Blöcke zerlegt.

Block 0:

Die Geschichte der Retrieval-Augmented Generation


In [ ]:
def ueberlappung(a: str, b: str) -> str:
    """Längster Text, mit dem a endet und b beginnt ('' = keine Überlappung)."""
    for n in range(min(len(a), len(b)), 0, -1):
        if a[-n:] == b[:n]:
            return a[-n:]
    return ""


# Das erste Paar suchen, das sich wirklich überlappt, und es zeigen.
for i in range(len(bloecke) - 1):
    geteilt = ueberlappung(bloecke[i], bloecke[i + 1])
    if geteilt:
        print(f"Block {i} endet mit:      ...{geteilt}")
        print()
        print(f"Block {i + 1} beginnt mit:  {geteilt}...")
        print()
        print(f"=> {len(geteilt)} Zeichen geteilt (angefragt: max. {UEBERLAPPUNG})")
        break
else:
    print("Kein Paar überlappt sich — BLOCK_GROESSE ist größer als die Absätze.")

# Überblick: bei wie vielen Nachbarpaaren klappt es überhaupt?
laengen = [len(ueberlappung(bloecke[i], bloecke[i + 1])) for i in range(len(bloecke) - 1)]
print()
print(f"{sum(1 for x in laengen if x)} von {len(laengen)} Nachbarpaaren überlappen sich, "
      f"im Schnitt {sum(laengen) / len(laengen):.0f} Zeichen.")
print("Die Nullen sind die Absatzgrenzen — dort trennt der Splitter ohne Überlappung.")


## Schritt 3: Embeddings berechnen und im FAISS-Index speichern

Ein einziger API-Aufruf bettet alle Blöcke auf einmal ein. Das
Embedding-Modell wandelt jeden Block in einen Vektor um; semantisch
ähnliche Texte bekommen ähnliche Vektoren.

`IndexFlatL2` ist der einfachste FAISS-Index: Er hält alle Vektoren
unkomprimiert im Arbeitsspeicher und vergleicht per L2-Distanz. FAISS kennt
nur Vektoren und ihre Position (0, 1, 2, ...) — die Texte selbst merken wir
uns daneben in der Liste `bloecke`.

Damit ist die **Indexierungsphase** abgeschlossen — ab hier läuft alles pro Frage.

In [11]:
antwort = client.embeddings.create(model=EMBEDDING_MODEL, input=bloecke)
vektoren = np.array([d.embedding for d in antwort.data], dtype="float32")
print(f"Embedding-Matrix: {vektoren.shape[0]} Blöcke × {vektoren.shape[1]} Dimensionen")

index = faiss.IndexFlatL2(vektoren.shape[1])
index.add(vektoren)
print(f"FAISS-Index mit {index.ntotal} Vektoren aufgebaut.")

Embedding-Matrix: 198 Blöcke × 1024 Dimensionen
FAISS-Index mit 198 Vektoren aufgebaut.


## Schritt 4: Retrieval — die ähnlichsten Blöcke zur Frage finden

Die Frage wird mit **demselben** Embedding-Modell eingebettet, FAISS liefert
die Positionen der `TOP_K` nächsten Vektoren plus deren Distanzen.
Der Score ist eine L2-Distanz: **kleiner bedeutet ähnlicher**.

In [12]:
def bloecke_suchen(frage: str) -> list[str]:
    """Die TOP_K ähnlichsten Blöcke zur Frage finden (kleiner Score = ähnlicher)."""
    einbettung = client.embeddings.create(model=EMBEDDING_MODEL, input=[frage])
    frage_vektor = np.array([einbettung.data[0].embedding], dtype="float32")
    distanzen, positionen = index.search(frage_vektor, min(TOP_K, index.ntotal))

    treffer = []
    print("Gefundene Blöcke (Score: kleiner = ähnlicher):")
    for score, position in zip(distanzen[0], positionen[0]):
        treffer.append(bloecke[position])
        vorschau = bloecke[position].replace("\n", " ")[:80]
        print(f"  [{score:.3f}] {vorschau}...")
    return treffer


treffer = bloecke_suchen("Wer hat den Begriff RAG geprägt?")

Gefundene Blöcke (Score: kleiner = ähnlicher):
  [0.687] Warum RAG? Die Kernprobleme, die es löst...
  [0.770] Lösung bereit: RAG wurde fast über Nacht zum...
  [0.800] RAG adressiert vier Grundprobleme großer...


## Schritt 5: Generation — das Chat-Modell antwortet nur aus dem Kontext

Die gefundenen Blöcke werden zu einem Kontext zusammengefügt. Eine
Nachrichtenliste und ein einziger API-Aufruf: Die System-Nachricht legt die
Spielregeln fest ("antworte NUR aus dem Kontext"), die User-Nachricht
enthält Kontext und Frage (per f-String eingesetzt). Der Antworttext steckt
in `choices[0]`.

In [13]:
def frage_beantworten(frage: str) -> str:
    """Abfragephase: passende Blöcke suchen und daraus eine Antwort erzeugen."""
    kontext = "\n\n---\n\n".join(bloecke_suchen(frage))

    antwort = client.chat.completions.create(
        model=CHAT_MODEL,
        # temperature=0: möglichst faktentreue, reproduzierbare Antworten -
        # genau das will man bei Q&A über eine feste Wissensbasis.
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Du bist ein Tutor für die Geschichte der RAG-Technik. "
                    "Beantworte die Frage AUSSCHLIESSLICH mit dem gelieferten "
                    "Kontext. Steht die Antwort nicht im Kontext, sage das "
                    "offen. Antworte auf Deutsch, kurz und präzise."
                ),
            },
            {"role": "user", "content": f"Kontext:\n{kontext}\n\nFrage: {frage}"},
        ],
    )
    return antwort.choices[0].message.content


print(frage_beantworten("Wer hat den Begriff RAG geprägt?"))

Gefundene Blöcke (Score: kleiner = ähnlicher):
  [0.687] Warum RAG? Die Kernprobleme, die es löst...
  [0.770] Lösung bereit: RAG wurde fast über Nacht zum...
  [0.800] RAG adressiert vier Grundprobleme großer...
Der Kontext liefert keine Informationen darüber, wer den Begriff RAG geprägt hat.


## Selbst ausprobieren

Stelle eigene Fragen! Interessant ist auch der Gegentest: Bei einer Frage,
deren Antwort **nicht** im Dokument steht, soll das Modell das offen
zugeben — genau das verlangt die System-Nachricht, und es verhindert
Halluzinationen.

In [ ]:
# Gegentest: Die Antwort steht nicht im Dokument.
print(frage_beantworten("Was ist die Hauptstadt von Frankreich?"))

In [ ]:
meine_frage = "Wie funktionierte RAG im Original-Paper?"
print(frage_beantworten(meine_frage))

## Experimente

- `TOP_K` erhöhen oder auf 1 senken — wie ändern sich Kontext und Antwort?
- `BLOCK_GROESSE` / `UEBERLAPPUNG` ändern und ab Schritt 2 neu ausführen —
  wie viele Blöcke entstehen, wie ändern sich die Scores?
- `temperature` erhöhen — werden die Antworten kreativer (und ungenauer)?
- Mit `BASE_URL` in der `.env` auf einen lokalen Ollama-Server umschalten.